# YatraLok — AI/ML Tourist Risk Prediction Model Training
### Member 4: Geo-Fencing + AI/ML Developer

> **Notice:** This AI model provides risk level predictions (`LOW`, `MEDIUM`, `HIGH`) to assist police authorities in prioritizing attention. The primary SOS mechanism operates independently and the final response decision remains with human officers.

## Step 1: Import Libraries and Configure Environment

In [ ]:
import numpy as np
import pandas as pd
import joblib
import os

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

## Step 2: Load the Dataset and Inspect Schema

In [ ]:
DATASET_PATH = 'data/dataset.csv' if os.path.exists('data/dataset.csv') else 'dataset.csv'
df = pd.read_csv(DATASET_PATH)

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())
print("\nClass distribution:")
print(df['risk_level'].value_counts())

## Step 3: Define 10 Model Features and Split by Trip Groups

In [ ]:
FEATURE_COLS = [
    'distance_from_danger_zone',
    'distance_from_restricted_zone',
    'speed',
    'time_of_day',
    'duration_in_remote_area',
    'movement_frequency',
    'route_deviation',
    'stationary_duration',
    'zone_entry_history',
    'previous_incident_context'
]
TARGET_COL = 'risk_level'

# Split dataset preserving whole-trip partitions
train_df = df[df['dataset_split'] == 'train'].copy()
val_df = df[df['dataset_split'] == 'validation'].copy()
test_df = df[df['dataset_split'] == 'test'].copy()

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
X_val, y_val = val_df[FEATURE_COLS], val_df[TARGET_COL]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET_COL]

print(f"Train records: {len(X_train)} | Val records: {len(X_val)} | Test records: {len(X_test)}")

## Step 4: Feature Scaling (StandardScaler fitted on Train only)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed successfully!")

## Step 5: Model 1 — Logistic Regression Baseline

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

lr_val_preds = lr_model.predict(X_val_scaled)
lr_acc = accuracy_score(y_val, lr_val_preds)
lr_f1 = f1_score(y_val, lr_val_preds, average='weighted')

print(f"Logistic Regression -> Val Accuracy: {lr_acc:.4f} | Weighted F1: {lr_f1:.4f}")

## Step 6: Model 2 — Decision Tree Classifier Baseline

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=8, random_state=42, class_weight='balanced')
dt_model.fit(X_train, y_train)

dt_val_preds = dt_model.predict(X_val)
dt_acc = accuracy_score(y_val, dt_val_preds)
dt_f1 = f1_score(y_val, dt_val_preds, average='weighted')

print(f"Decision Tree -> Val Accuracy: {dt_acc:.4f} | Weighted F1: {dt_f1:.4f}")

## Step 7: Model 3 — Random Forest Classifier (Ensemble)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, class_weight='balanced', n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_val_preds = rf_model.predict(X_val)
rf_acc = accuracy_score(y_val, rf_val_preds)
rf_f1 = f1_score(y_val, rf_val_preds, average='weighted')

print(f"Random Forest -> Val Accuracy: {rf_acc:.4f} | Weighted F1: {rf_f1:.4f}")

## Step 8: Final Model Evaluation on Unseen Test Set

In [ ]:
# Selecting Random Forest as the best performing model
best_model = rf_model
test_preds = best_model.predict(X_test)

acc = accuracy_score(y_test, test_preds)
prec = precision_score(y_test, test_preds, average='weighted')
rec = recall_score(y_test, test_preds, average='weighted')
f1 = f1_score(y_test, test_preds, average='weighted')
cm = confusion_matrix(y_test, test_preds, labels=['LOW', 'MEDIUM', 'HIGH'])

print("=== FINAL TEST EVALUATION RESULTS ===")
print(f"Accuracy  : {acc * 100:.2f}%")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")
print("\nConfusion Matrix (LOW, MEDIUM, HIGH):")
print(cm)
print("\nDetailed Classification Report:")
print(classification_report(y_test, test_preds))

## Step 9: Save Trained Model and Preprocessing Artifacts

In [ ]:
MODEL_SAVE_PATH = 'model.pkl'
model_payload = {
    'model': best_model,
    'features': FEATURE_COLS,
    'classes': best_model.classes_.tolist()
}

joblib.dump(model_payload, MODEL_SAVE_PATH)
print(f"Model successfully saved to {MODEL_SAVE_PATH}!")

## Step 10: Live Inference Test & Human-Explainable Output

In [ ]:
# Sample incoming tourist data
sample_input = pd.DataFrame([{
    'distance_from_danger_zone': 45.0,
    'distance_from_restricted_zone': 120.0,
    'speed': 2.5,
    'time_of_day': 23,
    'duration_in_remote_area': 12.0,
    'movement_frequency': 0.8,
    'route_deviation': 250.0,
    'stationary_duration': 0.0,
    'zone_entry_history': 1.0,
    'previous_incident_context': 450.0
}])

pred_level = best_model.predict(sample_input)[0]
pred_probs = best_model.predict_proba(sample_input)[0]
class_indices = {cls: idx for idx, cls in enumerate(best_model.classes_)}
risk_score = float(pred_probs[class_indices.get(pred_level, 0)])

print("Prediction Result:")
print({
    'risk_score': round(risk_score, 2),
    'risk_level': pred_level,
    'reason': 'Tourist movement is close to restricted perimeter and shows unusual route deviation.'
})